In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [31]:
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [32]:
import os
from collections import Counter
from sklearn.model_selection import train_test_split

In [33]:
dataset_path = "/kaggle/input/datasets/yidazhang07/bridge-cracks-image"

In [34]:
for root, dirs, files in os.walk(dataset_path):
    print(dirs)

['Bridge_Crack_Image', 'CrackForest', 'Magnetic-Tile-Defect', 'DeepPCB']
['DBCC_Training_Data_Set']
['val', 'train']
[]
[]
['groundTruth', 'seg', 'image']
[]
[]
['image']
[]
['MT_Fray', 'MT_Break', 'MT_Crack', 'MT_Free', 'MT_Blowhole']
['Imgs']
[]
['Imgs']
[]
['Imgs']
[]
['Imgs']
[]
['Imgs']
[]
['fig', 'PCBData', 'tools', 'evaluation']
['tools', 'result']
[]
[]
['group12300', 'group00041', 'group50600', 'group77000', 'group12100', 'group90100', 'group12000', 'group13000', 'group20085', 'group44000', 'group92000']
['12300_not', '12300']
[]
[]
['00041_not', '00041']
[]
[]
['50600_not', '50600']
[]
[]
['77000_not', '77000']
[]
[]
['12100_not', '12100']
[]
[]
['90100_not', '90100']
[]
[]
['12000_not', '12000']
[]
[]
['13000', '13000_not']
[]
[]
['20085', '20085_not']
[]
[]
['44000_not', '44000']
[]
[]
['92000_not', '92000']
[]
[]
['examples', 'PCBAnnotationTool']
['grouptest']
['test']
[]
['images']
[]
['gt']
[]


In [35]:
import os
image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

data = []

for class_name in os.listdir(dataset_path):
    #print(class_name)

    class_folder = os.path.join(dataset_path, class_name)
    # print(class_folder)

    if os.path.isdir(class_folder):

        for img in os.listdir("/kaggle/input/datasets/yidazhang07/bridge-cracks-image/Bridge_Crack_Image/DBCC_Training_Data_Set/train"):
            # print(os.listdir(class_folder))
            

            if img.lower().endswith(image_extensions):

                data.append({
                    "filepath": os.path.join(class_folder, img),
                    "label": class_name
                })

df = pd.DataFrame(data)

## DataFrame Information About the dataset

In [36]:
df.info

<bound method DataFrame.info of                                                  filepath               label
0       /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
1       /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
2       /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
3       /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
4       /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
...                                                   ...                 ...
199995  /kaggle/input/datasets/yidazhang07/bridge-crac...             DeepPCB
199996  /kaggle/input/datasets/yidazhang07/bridge-crac...             DeepPCB
199997  /kaggle/input/datasets/yidazhang07/bridge-crac...             DeepPCB
199998  /kaggle/input/datasets/yidazhang07/bridge-crac...             DeepPCB
199999  /kaggle/input/datasets/yidazhang07/bridge-crac...             DeepPCB

[200000 rows x 2 columns]>

## Totall Numbers of Images Avaible in Dataset

In [37]:
#totall images 
totall_images=len(df)
print(totall_images)

200000


## Numbers of Classes in Dataset

In [38]:
print(df["label"].unique())

['Bridge_Crack_Image' 'CrackForest' 'Magnetic-Tile-Defect' 'DeepPCB']


In [39]:
print(df.head(4))

                                            filepath               label
0  /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
1  /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
2  /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image
3  /kaggle/input/datasets/yidazhang07/bridge-crac...  Bridge_Crack_Image


In [40]:
print("Total Classes:", df["label"].nunique())

Total Classes: 4


## Count Images Per Class

In [41]:
class_count=df["label"].value_counts()
print(class_count)

label
Bridge_Crack_Image      50000
CrackForest             50000
Magnetic-Tile-Defect    50000
DeepPCB                 50000
Name: count, dtype: int64


## Percentage of Each Class in Dataset

In [42]:
percentage=df["label"].value_counts(normalize=True)*100
print(percentage)

label
Bridge_Crack_Image      25.0
CrackForest             25.0
Magnetic-Tile-Defect    25.0
DeepPCB                 25.0
Name: proportion, dtype: float64


## Is Dataset is balance or unbalance

In [43]:
check_balance=class_count.max() / class_count.min() 

if check_balance < 1.5:
    print(check_balance)
    print("Dataset is Balanced")
else :
    print(check_balance)
    print("Dataset is Unbalanced")

1.0
Dataset is Balanced


## Image Extension

## Split Dataset (73%, 17%, 10%)

In [45]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.27,
    stratify=df["label"],
    random_state=42
)

In [46]:
test_df, val_df = train_test_split(
    temp_df,
    test_size=10/27,
    stratify=temp_df["label"],
    random_state=42
)

In [47]:
print("Training:", len(train_df))
print("Testing :", len(test_df))
print("Validation:", len(val_df))

Training: 146000
Testing : 34000
Validation: 20000


In [ ]:
print("Train")
print(train_df["label"].value_counts())

print()

print("Test")
print(test_df["label"].value_counts())

print()

print("Validation")
print(val_df["label"].value_counts())

In [49]:
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
val_df.to_csv("validation.csv", index=False)